# Agent Loop

In [2]:
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# A normal python function. The docstring (text in triple quotes) explains it to the model.
def get_word_length(word : str) -> str:
    """Return the number of letters in a word."""
    return len(word)

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="How many letters in the word 'agentic'",
    config=types.GenerateContentConfig(tools=[get_word_length])
)    

print(response.text)


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


There are 7 letters in the word 'agentic'.


In [7]:
import os
from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])

# the tool: a simple calculator
def calculator(expression):
    allowed = "1234567890+-/*()." 
    for ch in expression:
        if ch not in allowed:
            return "ERROR: illegal character"
    return str(eval(expression))

# Rules for the model.
system_prompt = """
You are an agent with a calculator.
To use the calculator, reply with exactly one line: CALC <math expression>
When you know the final answer, reply with exactly one line: ANSWER <answer>
"""

messages = [
    {"role":"system","content":system_prompt},
    {"role":"user","content":"What is (15*4) +128/8?"}
]

for step in range(5):
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",messages=messages,temperature=0
    )
    reply = response.choices[0].message.content.strip()
    print("Model says: ",reply)
    messages.append({"role":"assistant","content":reply})

    if reply.startswith("ANSWER:"):
        break
    
    if reply.startswith("CALC:"):
        expression = reply.replace("CALC:","").strip()
        result = calculator(expression=expression)
        print("Tool result: ",result)
        messages.append({"role":"user","content":"Tool result: "+result})


Model says:  CALC (15*4)+128/8
Model says:  CALC (15*4)+128/8
ANSWER 76
Model says:  ANSWER 76
Model says:  ANSWER 76
Model says:  ANSWER 76


# Tool Design

In [11]:
# bad tool vs good tool

import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

#BAD tool
def tool1(a:str) -> str:
    """Does stuff."""
    print(">>> tool1 was called")
    return "31 C and sunny"

# Good tool
def get_Weather(city: str) -> str:
    """Get the current weather for a city. Example input: 'chennai'"""
    print(">>> get_weather was called")
    return "31 C and sunny"

question = "What is the weather in Chennai right now?"

for tool_to_use in [[tool1], [get_Weather]]:
    print("---Trying with: ",tool_to_use[0].__name__)
    response = client.models.generate_content(
        model = "gemini-3.5-flash-lite",
        contents = question,
        config = types.GenerateContentConfig(tools=tool_to_use)
    )
    print("Answer: ",response.text)
    print()


---Trying with:  tool1
Answer:  I do not have access to real-time weather information. Please check a live weather website or app to find the current conditions in Chennai.

---Trying with:  get_Weather
>>> get_weather was called
Answer:  The current weather in Chennai is 31°C and sunny.



In [13]:
import os
from groq import Groq
client = Groq(api_key = os.environ["GROQ_API_KEY"])

def divide(a,b):
    if b==0:
        return "ERROR: cannot divide by zero. Use a non-zero second number."
    return str(a/b)
result = divide(10,0)
print("Tool result: ",result)

messages = [
     {"role": "user", "content": "Divide 10 by 0 using the tool."},
    {"role": "assistant", "content": "I will call divide(10, 0)."},
    {"role": "user", "content": "Tool result: " + result + "\nExplain to the user what went wrong."},
]

response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=messages,
)

print("Agent: ",response.choices[0].message.content)


Tool result:  ERROR: cannot divide by zero. Use a non-zero second number.
Agent:  The tool attempted to perform the operation **10 ÷ 0**, but division by zero is mathematically undefined. In most programming languages (and in the tool you’re using), trying to divide by zero triggers an error because there is no number that you can multiply by 0 to get 10. Consequently, the tool returned the message:

```
ERROR: cannot divide by zero. Use a non-zero second number.
```

So the operation cannot be completed as written. If you need a result, you’ll have to choose a non‑zero divisor (e.g., `divide(10, 2)` would give `5`). If you’re exploring concepts like limits or asymptotic behavior, those are handled differently and aren’t performed by the simple arithmetic tool. Let me know how you’d like to proceed!


# 3. Context engineering

In [14]:
import os
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])
messages = [{"role":"system","content":"You are a short, friendly assistant."}]

user_inputs = [
    "My name is kumar",
    "I live in London",
    "What is my name ans where do i live?"
]

for text in user_inputs:
    messages.append({"role":"user","content":text})
    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",messages=messages
    )

    answer = response.choices[0].message.content
    messages.append({"role":"assistant","content":answer})
    print("User: ",text)
    print("Agent: ",answer)
    print()



User:  My name is kumar
Agent:  Nice to meet you, Kumar! How can I help you today?

User:  I live in London
Agent:  London’s a great city! Is there anything you’d like to know or discuss about it?

User:  What is my name ans where do i live?
Agent:  Your name is Kumar, and you live in London.



In [15]:
import os
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

def trim(messages, keep_last=4):
    system_message = messages[0] # always keep the rules
    recent = messages[-keep_last:]
    return[system_message] + recent

messages = [{"role":"system","content":"You are a helpful assistant"}]
for i in range(1,9):
     messages.append({"role": "user", "content": "Old message number " + str(i)})
messages.append({"role": "user", "content": "Remember: the secret code is 4321."})
messages.append({"role": "user", "content": "What was the secret code?"})

print("Messages before trimming: ",len(messages))
messages = trim(messages=messages)
print("messages after trimming: ",len(messages))

response = client.chat.completions.create(model="openai/gpt-oss-120b",messages=messages)
print("Agent: ",response.choices[0].message.content)


Messages before trimming:  11
messages after trimming:  5
Agent:  The secret code you mentioned is **4321**.


In [16]:
import os
from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

# our small knowledge base 
knowledge = {
    "refund": "refunds are processed within 5 business days.",
    "shipping": "Shipping takes 3 to 7 across india.",
    "warranty" : "All products have a 1 year warranty."
}

question = "How long does shipping take?"

# find only facts that match words in the question
found = []
for keyword in knowledge:
    if keyword in question.lower():
        found.append(knowledge[keyword])

context = "\n".join(found)
print("Facts added to context: ",context)

prompt = "Answer using only this information: \n "+context +"\n\nQuestion:"+question
response = client.models.generate_content(model="gemini-3.5-flash-lite",contents=prompt)
print("answer: ",response.text)




Facts added to context:  Shipping takes 3 to 7 across india.
answer:  Shipping takes 3 to 7.
